# 02 – SimPy basics
Small step-by-step experiments that build up to the full line simulation in `src/simulation.py`.

### One station, fixed times
A single operator at op 7 (the slowest operation, SMV 0.90 min). Shirts arrive every 0.60 min (the takt time, $480 / 800$).
- `shirt()` is one garment: it requests the operator, waits if busy, then holds the operator for `SERVICE_TIME`.
- `arrivals()` releases a new shirt every `ARRIVAL_GAP` minutes, forever.

Because arrivals (0.60 min) are faster than service (0.90 min), the queue should grow.

In [1]:
import random

import pandas as pd
import simpy

ARRIVAL_GAP = 0.60   # min between arriving shirts (= takt)
SERVICE_TIME = 0.90  # min per shirt at op 7, 1 operator (PLACEHOLDER SMV)
SIM_TIME = 60        # simulate one hour


def shirt(env, name, station, log):
    """One shirt: arrive, wait for the operator, get sewn, leave."""
    arrive = env.now
    with station.request() as req:
        yield req                          # wait until the operator is free
        start = env.now
        yield env.timeout(SERVICE_TIME)    # sewing takes SERVICE_TIME minutes
    log.append((name, arrive, start, env.now))


def arrivals(env, station, log):
    """Release a new shirt into the line every ARRIVAL_GAP minutes."""
    i = 0
    while True:
        env.process(shirt(env, i, station, log))
        i += 1
        yield env.timeout(ARRIVAL_GAP)

### Run one hour
Run the model for 60 min and log each shirt's times.
- **wait** = start − arrive (time spent in the queue).
- Expected finished ≈ $60 / 0.90 \approx 66$ shirts, while ≈ $60 / 0.60 = 100$ arrive, so about 33 are still waiting at the end and the wait time keeps increasing.

In [2]:
env = simpy.Environment()
station = simpy.Resource(env, capacity=1)
log = []
env.process(arrivals(env, station, log))
env.run(until=SIM_TIME)

res = pd.DataFrame(log, columns=["shirt", "arrive", "start", "finish"])
res["wait"] = res["start"] - res["arrive"]
print("finished in 60 min:", len(res))
print(res.tail(3).round(2))
print("shirts waiting at end:", len(station.queue))

finished in 60 min: 66
    shirt  arrive  start  finish  wait
63     63    37.8   56.7    57.6  18.9
64     64    38.4   57.6    58.5  19.2
65     65    39.0   58.5    59.4  19.5
shirts waiting at end: 33


### Add randomness and a choice of operators
Same station, wrapped in a function so it can be rerun with different settings:
- `capacity` = number of operators at the station.
- Service time = $\text{SMV} \times U(1 - v,\ 1 + v)$, i.e. uniformly random within ±`variability` of the SMV.
- `seed` fixes the random numbers so a run can be repeated exactly.

Utilization = share of paid time the operators were sewing:

$$\text{utilization} = \frac{\sum(\text{finish} - \text{start})}{\text{capacity} \times \text{sim time}}$$

In [3]:
def run_station(capacity: int, variability: float, seed: int, sim_time: float = 60) -> dict:
    """Simulate op 7 with random service times; return output, queue and utilization."""
    rng = random.Random(seed)
    env = simpy.Environment()
    station = simpy.Resource(env, capacity=capacity)
    log = []

    def shirt(name):
        arrive = env.now
        with station.request() as req:
            yield req
            start = env.now
            yield env.timeout(SERVICE_TIME * rng.uniform(1 - variability, 1 + variability))
        log.append((name, arrive, start, env.now))

    def arrivals():
        i = 0
        while True:
            env.process(shirt(i))
            i += 1
            yield env.timeout(ARRIVAL_GAP)

    env.process(arrivals())
    env.run(until=sim_time)
    res = pd.DataFrame(log, columns=["shirt", "arrive", "start", "finish"])
    busy = (res["finish"] - res["start"]).sum()
    return {"finished": len(res), "waiting_at_end": len(station.queue),
            "mean_wait": round(float((res["start"] - res["arrive"]).mean()), 2),
            "utilization": round(float(busy / (capacity * sim_time)), 3)}

### Compare 1 vs. 2 operators
Run every combination of operators (1, 2), variability (0, ±15%) and seed (1–3).
- 1 operator: capacity $1/0.90 = 1.11$ shirts/min < demand $1/0.60 = 1.67$ → overloaded, utilization ≈ 1, long queue.
- 2 operators: capacity $2/0.90 = 2.22$ shirts/min > demand → no queue, utilization ≈ $0.90 / (2 \times 0.60) = 0.75$.

The last line checks reproducibility: the same seed must give identical results.

In [4]:
rows = []
for cap in [1, 2]:
    for var in [0.0, 0.15]:
        for seed in [1, 2, 3]:
            rows.append({"operators": cap, "variability": var, "seed": seed,
                         **run_station(cap, var, seed)})
print(pd.DataFrame(rows).to_string(index=False))
print("same seed, same result:", run_station(1, 0.15, 1) == run_station(1, 0.15, 1))

 operators  variability  seed  finished  waiting_at_end  mean_wait  utilization
         1         0.00     1        66              33       9.75        0.990
         1         0.00     2        66              33       9.75        0.990
         1         0.00     3        66              33       9.75        0.990
         1         0.15     1        66              33       9.46        0.985
         1         0.15     2        65              34       9.96        0.994
         1         0.15     3        65              34       9.88        0.995
         2         0.00     1        99               0       0.00        0.742
         2         0.00     2        99               0       0.00        0.742
         2         0.00     3        99               0       0.00        0.742
         2         0.15     1        99               0       0.00        0.745
         2         0.15     2        99               0       0.00        0.754
         2         0.15     3        99 

### Two stations with a buffer (rack) between them
Station 1 → buffer → Station 2, one operator each, both with SMV = takt = 0.60 min. Station 1 always has cut parts.
- `simpy.Store(capacity=buffer_size)` is the rack.
- Station 1 is **blocked** when the rack is full (`put` waits).
- Station 2 is **starved** when the rack is empty (`get` waits).

Run for a full shift (480 min). Maximum possible output is $480 / 0.60 = 800$.

In [5]:
def run_two_stations(buffer_size: int, variability: float, seed: int,
                     sim_time: float = 480) -> dict:
    """Station 1 -> buffer -> Station 2; one operator each, SMV = takt = 0.60."""
    rng = random.Random(seed)
    env = simpy.Environment()
    buffer = simpy.Store(env, capacity=buffer_size)
    busy = {"s1": 0.0, "s2": 0.0}
    done = []

    def work_time():
        return 0.60 * rng.uniform(1 - variability, 1 + variability)

    def station1():
        while True:                        # cut parts always available
            t = work_time()
            yield env.timeout(t)
            busy["s1"] += t
            yield buffer.put("shirt")      # BLOCKED here if buffer is full

    def station2():
        while True:
            yield buffer.get()             # STARVED here if buffer is empty
            t = work_time()
            yield env.timeout(t)
            busy["s2"] += t
            done.append(env.now)

    env.process(station1())
    env.process(station2())
    env.run(until=sim_time)
    return {"output": len(done),
            "util_s1": round(busy["s1"] / sim_time, 3),
            "util_s2": round(busy["s2"] / sim_time, 3)}

### Effect of buffer size and variability
For each buffer size and variability, average output and Station 2 utilization over 10 seeds.
- With no variability, output is the same for any buffer.
- With variability, a small buffer causes more blocking/starving and lost output; a larger buffer absorbs the fluctuations.

In [6]:
rows = []
for buf in [1, 3, 10]:
    for var in [0.0, 0.15, 0.25]:
        runs = [run_two_stations(buf, var, seed) for seed in range(1, 11)]
        rows.append({"buffer": buf, "variability": var,
                     "mean_output": sum(r["output"] for r in runs) / len(runs),
                     "mean_util_s2": round(sum(r["util_s2"] for r in runs) / len(runs), 3)})
print(pd.DataFrame(rows).to_string(index=False))

 buffer  variability  mean_output  mean_util_s2
      1         0.00        798.0         0.998
      1         0.15        792.6         0.991
      1         0.25        784.5         0.981
      3         0.00        798.0         0.998
      3         0.15        795.7         0.995
      3         0.25        791.6         0.990
     10         0.00        798.0         0.998
     10         0.15        796.2         0.994
     10         0.25        794.5         0.994
